In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

import os
from datetime import datetime
import glob

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("="*80)
print("🚗 ПРЕДСКАЗАНИЕ ЦЕН АВТОМОБИЛЕЙ - ПОЛНЫЙ ПАЙПЛАЙН")
print("="*80)

🚗 ПРЕДСКАЗАНИЕ ЦЕН АВТОМОБИЛЕЙ - ПОЛНЫЙ ПАЙПЛАЙН


In [2]:
print("\n" + "="*80)
print("ШАГ 1: ЗАГРУЗКА ДАННЫХ")
print("="*80)

def load_data_from_folders_glob(base_path, filename="data.csv"):
    """
    Загружает все data.csv из подпапок base_path
    """
    csv_files = glob.glob(os.path.join(base_path, "**", filename), recursive=True)
    print(f"Найдено файлов: {len(csv_files)}")
    
    dataframes = []
    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)
            dataframes.append(df)
        except Exception as e:
            print(f"  Ошибка при загрузке {file_path}: {e}")
    
    if dataframes:
        X_train = pd.concat(dataframes, ignore_index=True)
        print(f"\n✅ Итоговый размер X_train: {X_train.shape}")
        return X_train
    else:
        print("❌ Файлы data.csv не найдены!")
        return pd.DataFrame()

# Загружаем данные
base_directory = "./"  # Путь к папке с 48 подпапками
X_train = load_data_from_folders_glob(base_directory)
y_train = pd.read_csv('./y_train_base.csv')
X_test = pd.read_csv('./X_test_base.csv')

print("\n" + "="*80)
print("РАЗМЕРЫ ДАННЫХ ПОСЛЕ ЗАГРУЗКИ")
print("="*80)
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")


ШАГ 1: ЗАГРУЗКА ДАННЫХ
Найдено файлов: 18

✅ Итоговый размер X_train: (18002, 22)

РАЗМЕРЫ ДАННЫХ ПОСЛЕ ЗАГРУЗКИ
X_train: (18002, 22)
y_train: (8340, 2)
X_test:  (8341, 22)


In [4]:
X_train.head()

,car_id,Бренд,Год выпуска,Модель,Тип машины,Полное название,Исползование,КПП,Двигатель,Привод,Топливо,Расход,Пробег,Цвет,Локация,Количество цилиндров,Тип кузова,Двери,Количество кресел,Оценка эксперта,Количество владельцев,Предложение
0,04d6b325-5483-43ab-a97b-5b0bcc23c3ca,KIA,2020.0000,SPORTAGE,SUV,2020 KIA SPORTAGE S (FWD),USED,Automatic,"4 cyl, 2 L",Front,UNLEADED,7.9 L / 100 km,58927,Silver / -,"WELSHPOOL, WA",4 cyl,SUV,4 Doors,5 Seats,7.0000,3.0000,29379.0000
1,59b671dd-fda2-4d74-97bc-6168f604fb00,SUBARU,2015.0000,FORESTER,USED DEALER AD,2015 SUBARU FORESTER,USED,-,-,Other,UNLEADED,-,33620,White / Black,"OSBORNE PARK, WA",-,WAGON,NaN,NaN,5.0000,2.0000,27385.0000
2,008e6e1e-2134-48ff-bd78-f63673fa511e,MERCEDES-BENZ,2013.0000,A250,HATCHBACK,2013 MERCEDES-BENZ A250 SPORT,USED,Automatic,"4 cyl, 2 L",Front,PREMIUM,6.6 L / 100 km,67434,Red / -,"ROCKLEA, QLD",4 cyl,HATCHBACK,5 Doors,5 Seats,7.0000,4.0000,21066.0000
3,4f75aa1e-0ae6-46d5-9f21-df17e8646a1f,GWM,2023.0000,HAVAL,KEDRON HAVAL & GREAT WALL ONLINE,2023 GWM HAVAL H6GT ULTRA (4WD),NEW,Automatic,"4 cyl, 2 L",4WD,UNLEADED,8.4 L / 100 km,15,Red / Charcoal,"KEDRON, QLD",4 cyl,SUV,4 Doors,5 Seats,4.0000,4.0000,41822.0000
4,c6ca40d5-eacc-4ae8-8b92-7092dd8db378,KIA,2015.0000,CARNIVAL,WAGON,2015 KIA CARNIVAL S,USED,Automatic,"4 cyl, 2.2 L",Front,DIESEL,7.7 L / 100 km,74402,White / -,"NOWRA, NSW",4 cyl,WAGON,4 Doors,8 Seats,8.0000,2.0000,27202.0000


In [5]:
y_train.head()

,car_id,Цена
0,099ec55b-6322-4a9f-8d5e-0bed830c3023,28777
1,25e3e231-5efc-4032-abe9-794c0cba8295,62999
2,54f3b009-8e37-49cd-905a-9637d6bf135c,28999
3,b361a8ff-af76-40cc-9f54-811b6bfd5eb5,33950
4,671c0cce-667a-4f44-87e7-fe999e17e55e,77888
